# Tutorial 03: High-Throughput Parameter Scanning

This tutorial demonstrates how to use the ParameterScanner for materials screening.

**Learning objectives:**
- Set up parameter sweeps over ΔQ and ΔE
- Run parallel calculations
- Visualize 2D heatmaps
- Save and load scan results
- Identify optimal material parameters

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from carriercapture.analysis.parameter_scan import (
    ParameterScanner,
    ScanParameters,
    ScanResult,
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Define Scan Parameters

We'll scan over:
- ΔQ: 0 to 25 amu^0.5·Å (configuration coordinate shift)
- ΔE: 0 to 2.5 eV (energy difference)

Using harmonic potentials with ℏω = 8 meV at T = 300 K.

In [ ]:
params = ScanParameters(
    dQ_range=(0, 25, 25),   # (min, max, n_points)
    dE_range=(0, 2.5, 10),  # (min, max, n_points)
    hbar_omega_i=0.008,     # Initial state phonon (eV)
    hbar_omega_f=0.008,     # Final state phonon (eV)
    temperature=300.0,      # Temperature (K)
    volume=1e-21,           # Supercell volume (cm³)
    degeneracy=1,
    nev_initial=180,        # Eigenvalues for initial state
    nev_final=60,           # Eigenvalues for final state
)

print(f"Scan configuration:")
print(f"  ΔQ: {params.dQ_range[2]} points from {params.dQ_range[0]} to {params.dQ_range[1]}")
print(f"  ΔE: {params.dE_range[2]} points from {params.dE_range[0]} to {params.dE_range[1]}")
print(f"  Total: {params.dQ_range[2] * params.dE_range[2]} calculations")

## 2. Run Parameter Scan

The scanner automatically:
- Creates harmonic potentials for each grid point
- Solves Schrödinger equations
- Calculates capture coefficients
- Handles convergence issues gracefully

In [ ]:
# Create scanner
scanner = ParameterScanner(params, verbose=True)

# Run scan (serial for demonstration, use n_jobs=-1 for all cores)
results = scanner.run_harmonic_scan(
    n_jobs=1,           # Use 1 for serial, -1 for all cores
    show_progress=True,
)

# Check results
n_success = np.sum(~np.isnan(results.capture_coefficients))
n_total = results.capture_coefficients.size
print(f"\nSuccessful calculations: {n_success}/{n_total} ({100*n_success/n_total:.1f}%)")

## 3. Visualize Results: 2D Heatmap

In [ ]:
# Create heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Use log scale for better visualization
Z = np.log10(results.capture_coefficients + 1e-30)

im = ax.pcolormesh(
    results.dE_grid,
    results.dQ_grid,
    Z,
    cmap='viridis',
    shading='auto',
)

ax.set_xlabel('ΔE (eV)', fontsize=14)
ax.set_ylabel('ΔQ (amu$^{0.5}$·Å)', fontsize=14)
ax.set_title('Capture Coefficient at 300 K', fontsize=16)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('log₁₀(C) [cm³/s]', fontsize=12)

plt.tight_layout()
plt.show()

## 4. Identify Optimal Parameters

Find material configurations with desired capture rates.

In [ ]:
# Find maximum capture coefficient
valid_mask = ~np.isnan(results.capture_coefficients)
valid_coeffs = results.capture_coefficients[valid_mask]

if len(valid_coeffs) > 0:
    # Maximum
    max_idx = np.unravel_index(np.nanargmax(results.capture_coefficients), results.capture_coefficients.shape)
    max_dQ = results.dQ_grid[max_idx[0]]
    max_dE = results.dE_grid[max_idx[1]]
    max_C = results.capture_coefficients[max_idx]
    
    print(f"Maximum capture coefficient:")
    print(f"  C = {max_C:.3e} cm³/s")
    print(f"  at ΔQ = {max_dQ:.1f} amu^0.5·Å, ΔE = {max_dE:.2f} eV")
    
    # Minimum
    min_idx = np.unravel_index(np.nanargmin(results.capture_coefficients), results.capture_coefficients.shape)
    min_dQ = results.dQ_grid[min_idx[0]]
    min_dE = results.dE_grid[min_idx[1]]
    min_C = results.capture_coefficients[min_idx]
    
    print(f"\nMinimum capture coefficient:")
    print(f"  C = {min_C:.3e} cm³/s")
    print(f"  at ΔQ = {min_dQ:.1f} amu^0.5·Å, ΔE = {min_dE:.2f} eV")
    
    # Statistics
    print(f"\nStatistics:")
    print(f"  Mean: {valid_coeffs.mean():.3e} cm³/s")
    print(f"  Std:  {valid_coeffs.std():.3e} cm³/s")
    print(f"  Range: {valid_coeffs.max()/valid_coeffs.min():.1e}×")

## 5. Cross-Sections: Effect of ΔQ and ΔE

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Fix ΔE = 0.5 eV, vary ΔQ
dE_idx = np.argmin(np.abs(results.dE_grid - 0.5))
C_vs_dQ = results.capture_coefficients[:, dE_idx]

ax1.semilogy(results.dQ_grid, C_vs_dQ, 'o-', linewidth=2, markersize=6)
ax1.set_xlabel('ΔQ (amu$^{0.5}$·Å)', fontsize=12)
ax1.set_ylabel('C (cm³/s)', fontsize=12)
ax1.set_title(f'ΔE = {results.dE_grid[dE_idx]:.2f} eV (fixed)', fontsize=14)
ax1.grid(True, alpha=0.3)

# Fix ΔQ = 10 amu^0.5·Å, vary ΔE
dQ_idx = np.argmin(np.abs(results.dQ_grid - 10.0))
C_vs_dE = results.capture_coefficients[dQ_idx, :]

ax2.semilogy(results.dE_grid, C_vs_dE, 's-', linewidth=2, markersize=6, color='C1')
ax2.set_xlabel('ΔE (eV)', fontsize=12)
ax2.set_ylabel('C (cm³/s)', fontsize=12)
ax2.set_title(f'ΔQ = {results.dQ_grid[dQ_idx]:.1f} amu$^{{0.5}}$·Å (fixed)', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save and Load Results

In [ ]:
# Save to NPZ format
results.save('scan_results.npz', format='npz')
print("Saved to scan_results.npz")

# Load back
loaded_results = ScanResult.load('scan_results.npz', format='npz')
print(f"Loaded: {loaded_results.dQ_grid.shape[0]}×{loaded_results.dE_grid.shape[0]} grid")

# Verify
assert np.allclose(loaded_results.capture_coefficients, results.capture_coefficients, equal_nan=True)
print("✓ Verification passed")

## 7. Physical Insights

From the parameter scan, we observe:

1. **ΔQ dependence**:
   - Larger ΔQ generally increases capture rate
   - But too large ΔQ can reduce overlap (Franck-Condon factors)

2. **ΔE dependence**:
   - Smaller ΔE leads to faster capture (lower barrier)
   - But ΔE affects thermal accessibility

3. **Optimal regions**:
   - High capture: moderate ΔQ (8-12), low ΔE (< 1 eV)
   - Low capture: small ΔQ (< 5) or large ΔE (> 2 eV)

## Summary

In this tutorial, we:
- ✓ Set up high-throughput parameter scans
- ✓ Ran 250 calculations systematically
- ✓ Visualized results as 2D heatmaps
- ✓ Identified optimal material parameters
- ✓ Analyzed trends in ΔQ and ΔE

**Next steps:**
- Use parallel execution (`n_jobs=-1`) for larger scans
- Scan over phonon frequencies (ℏω_i, ℏω_f)
- Apply to real material systems

## Exercise: 3D Parameter Space

Extend to scan over phonon frequency as well.

In [ ]:
# Exercise: Scan over phonon frequency
hw_values = [0.004, 0.008, 0.012]  # 4, 8, 12 meV

results_hw = {}
for hw in hw_values:
    params_hw = ScanParameters(
        dQ_range=(0, 25, 15),  # Coarser grid for speed
        dE_range=(0, 2.5, 8),
        hbar_omega_i=hw,
        hbar_omega_f=hw,
        temperature=300.0,
    )
    
    scanner_hw = ParameterScanner(params_hw, verbose=False)
    results_hw[hw] = scanner_hw.run_harmonic_scan(n_jobs=1, show_progress=False)
    print(f"Completed scan for ℏω = {hw*1000:.0f} meV")

# Compare at fixed point
print("\nCapture coefficient at ΔQ=10, ΔE=0.5:")
for hw in hw_values:
    res = results_hw[hw]
    i_dQ = np.argmin(np.abs(res.dQ_grid - 10))
    i_dE = np.argmin(np.abs(res.dE_grid - 0.5))
    C = res.capture_coefficients[i_dQ, i_dE]
    print(f"  ℏω = {hw*1000:.0f} meV: C = {C:.3e} cm³/s")